In [1]:
from datetime import datetime, timezone
from pathlib import Path
import sys

import joblib
import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from pathlib import Path
from datetime import datetime, timezone
import json
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [2]:
PROJECT_DIR = Path("..").resolve()
DEV_PATH = PROJECT_DIR / "data" / "processed" / "v3e1_dev_sep1_7.parquet"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_PATH = MODEL_DIR / "lightgbm_v4_t2_b_02_final.joblib"
# IMPORTANT: Replace with the real, verified V3-E1 holdout feature file.
TEST_PATH = PROJECT_DIR / "data" / "processed" / "v3e1_test_sep8_plus.parquet"

OUTPUT_DIR = PROJECT_DIR / "Results" / "final_test"
RESULTS_PATH = OUTPUT_DIR / "v4_t2_b_02_final_holdout_metrics.json"

print("Repository:", PROJECT_DIR)
print("Input:", DEV_PATH)
print("Output:", MODEL_PATH)

assert MODEL_PATH.is_file(), f"Missing frozen artifact: {MODEL_PATH}"
print("Model:", MODEL_PATH)
print("Test candidate:", TEST_PATH)
print("Results output:", RESULTS_PATH)

Repository: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection
Input: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\v3e1_dev_sep1_7.parquet
Output: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\models\lightgbm_v4_t2_b_02_final.joblib
Model: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\models\lightgbm_v4_t2_b_02_final.joblib
Test candidate: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\v3e1_test_sep8_plus.parquet
Results output: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\Results\final_test\v4_t2_b_02_final_holdout_metrics.json


# Freezing the selected configuration

In [3]:
MODEL_ID = "V4-T2-B-02"
FEATURE_VERSION = "V3-E1"
FINAL_THRESHOLD = 0.105592425529
EXPECTED_DEV_ROWS = 3_731_672
TARGET_COLUMN = "is_laundering"
TIMESTAMP_COLUMN = "transaction_timestamp"

In [4]:
NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
    "sender_tx_count_1h",
    "log_sender_amount_paid_sum_24h_same_currency",
    "log_sender_hours_since_prev_tx",
    "log_amount_paid_vs_sender_median",
    "receiver_seen_before",
    "sender_receiver_prior_tx_count",
    "sender_distinct_receivers_24h",
]
CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [5]:
MODEL_PARAMS = {
    "objective": "binary",
    "num_leaves": 63,
    "max_depth": 10,
    "min_child_samples": 500,
    "learning_rate": 0.03,
    "n_estimators": 500,
    "reg_alpha": 0.1,
    "reg_lambda": 2.0,
    "subsample": 0.9,
    "colsample_bytree": 0.75,
    "subsample_freq": 1,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
assert len(FEATURE_COLUMNS) == 17
print("Frozen model:", MODEL_ID)
print("Features:", len(FEATURE_COLUMNS))
print("Fixed threshold:", FINAL_THRESHOLD)


Frozen model: V4-T2-B-02
Features: 17
Fixed threshold: 0.105592425529


# Loading and validating the development data

In [6]:
if not DEV_PATH.is_file():
    raise FileNotFoundError(f"Missing development dataset: {DEV_PATH}")

dev_df = pd.read_parquet(DEV_PATH)
required = FEATURE_COLUMNS + [TARGET_COLUMN, TIMESTAMP_COLUMN]
missing = [column for column in required if column not in dev_df.columns]
if missing:
    raise ValueError(f"Missing V3-E1 columns: {missing}")
if len(dev_df) != EXPECTED_DEV_ROWS:
    raise ValueError(f"Expected {EXPECTED_DEV_ROWS:,} rows; got {len(dev_df):,}")

timestamps = pd.to_datetime(dev_df[TIMESTAMP_COLUMN], errors="raise")
if timestamps.isna().any() or not timestamps.is_monotonic_increasing:
    raise ValueError("Timestamps contain nulls or development rows are out of order")
if timestamps.min() < pd.Timestamp("2022-09-01") or timestamps.max() >= pd.Timestamp(
    "2022-09-08"
):
    raise ValueError("Only September 1–7, 2022 may appear in development data")
if dev_df[TARGET_COLUMN].isna().any() or not dev_df[TARGET_COLUMN].isin([0, 1]).all():
    raise ValueError("Target must contain only non-null 0/1 values")

X_dev = dev_df[FEATURE_COLUMNS]
y_dev = dev_df[TARGET_COLUMN].astype(int)
if y_dev.nunique() != 2:
    raise ValueError("Both classes must be represented")

print("Development transactions:", f"{len(dev_df):,}")
print("Fraud cases:", f"{y_dev.sum():,}")
print("Prevalence:", f"{y_dev.mean():.4%}")
print("Date range:", timestamps.min(), "to", timestamps.max())

Development transactions: 3,731,672
Fraud cases: 3,027
Prevalence: 0.0811%
Date range: 2022-09-01 00:00:00 to 2022-09-07 23:59:00


# Preprocessing

In [7]:
numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

final_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LGBMClassifier(**MODEL_PARAMS)),
    ]
)
print(final_pipeline.named_steps["model"].get_params())

{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.75, 'importance_type': 'split', 'learning_rate': 0.03, 'max_depth': 10, 'min_child_samples': 500, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 63, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.1, 'reg_lambda': 2.0, 'subsample': 0.9, 'subsample_for_bin': 200000, 'subsample_freq': 1, 'verbosity': -1}


# Refitting on all sept 1-7 development transactions

In [8]:
sample_weights = compute_sample_weight(class_weight="balanced", y=y_dev)
final_pipeline.fit(X_dev, y_dev, model__sample_weight=sample_weights)
print("Final development refit finished.")
print(
    "Number of boosting iterations:", final_pipeline.named_steps["model"].n_estimators_
)

Final development refit finished.
Number of boosting iterations: 500


In [9]:
artifact = {
    "model_id": MODEL_ID,
    "feature_version": FEATURE_VERSION,
    "pipeline": final_pipeline,
    "feature_columns": FEATURE_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "threshold": FINAL_THRESHOLD,
    "threshold_selection": "minimum historical per-fold OOF threshold for >=80% recall",
    "threshold_is_calibrated_probability": False,
    "train_start_inclusive": "2022-09-01",
    "train_end_exclusive": "2022-09-08",
    "training_rows": int(len(X_dev)),
    "training_fraud_cases": int(y_dev.sum()),
    "model_parameters": MODEL_PARAMS,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "versions": {
        "python": sys.version.split()[0],
        "sklearn": sklearn.__version__,
        "lightgbm": lightgbm.__version__,
        "pandas": pd.__version__,
    },
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)
print("Saved:", MODEL_PATH)
print("File size (MB):", round(MODEL_PATH.stat().st_size / 1_000_000, 2))

Saved: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\models\lightgbm_v4_t2_b_02_final.joblib
File size (MB): 3.51


# Smoke test inference

In [10]:
loaded = joblib.load(MODEL_PATH)
assert loaded["model_id"] == MODEL_ID
assert loaded["feature_columns"] == FEATURE_COLUMNS
assert loaded["threshold"] == FINAL_THRESHOLD

example = X_dev.iloc[:5]
original_scores = final_pipeline.predict_proba(example)[:, 1]
loaded_scores = loaded["pipeline"].predict_proba(example)[:, 1]
np.testing.assert_allclose(original_scores, loaded_scores, rtol=0, atol=1e-12)
assert np.isfinite(loaded_scores).all()
assert ((loaded_scores >= 0) & (loaded_scores <= 1)).all()

example_alerts = loaded_scores >= loaded["threshold"]
print("Successfully reloaded and checked saved artifact.")
print("Example risk scores:", loaded_scores)
print("Example binary flags:", example_alerts.astype(int))

Successfully reloaded and checked saved artifact.
Example risk scores: [5.50887211e-05 2.44564014e-04 2.48868316e-03 1.15730148e-04
 1.04310723e-04]
Example binary flags: [0 0 0 0 0]


In [11]:
artifact = joblib.load(MODEL_PATH)  # Only load trusted joblib files.
assert artifact["model_id"] == "V4-T2-B-02"
assert artifact["feature_version"] == "V3-E1"
assert np.isclose(artifact["threshold"], 0.105592425529, atol=1e-12)
assert len(artifact["feature_columns"]) == 17
assert artifact["training_rows"] == 3_731_672

FEATURE_COLUMNS = artifact["feature_columns"]
THRESHOLD = float(artifact["threshold"])
TARGET = "is_laundering"
TIMESTAMP = "transaction_timestamp"
print("Frozen model:", artifact["model_id"])
print("Feature schema:", FEATURE_COLUMNS)
print("Threshold:", THRESHOLD)

Frozen model: V4-T2-B-02
Feature schema: ['hour_of_day', 'day_of_week', 'is_weekend', 'same_currency_flag', 'same_bank_flag', 'log_amount_received', 'log_amount_paid', 'sender_tx_count_1h', 'log_sender_amount_paid_sum_24h_same_currency', 'log_sender_hours_since_prev_tx', 'log_amount_paid_vs_sender_median', 'receiver_seen_before', 'sender_receiver_prior_tx_count', 'sender_distinct_receivers_24h', 'receiving_currency', 'payment_currency', 'payment_format']
Threshold: 0.105592425529


# Untouched holdout and validate its shape

In [12]:
assert TEST_PATH.is_file(), (
    f"Test parquet not found: {TEST_PATH}. Locate the real V3-E1 holdout file. "
    "Do not substitute the development parquet."
)
assert not RESULTS_PATH.exists(), (
    "Final holdout metrics already exist. Do not rerun/re-tune against the test set."
)

test_df = pd.read_parquet(TEST_PATH)
required = FEATURE_COLUMNS + [TARGET, TIMESTAMP]
missing = [c for c in required if c not in test_df.columns]
assert not missing, f"Missing columns: {missing}"
assert len(test_df) > 0, "Empty holdout"

times = pd.to_datetime(test_df[TIMESTAMP], errors="raise")
assert times.notna().all()
assert times.min() >= pd.Timestamp("2022-09-08"), "Holdout includes development dates"
assert times.is_monotonic_increasing, "Holdout timestamps must be chronological"
assert test_df[TARGET].notna().all() and test_df[TARGET].isin([0, 1]).all()

X_test = test_df.loc[:, FEATURE_COLUMNS]
y_test = test_df[TARGET].astype(int)
assert y_test.nunique() == 2, "Need both classes for AP and ROC-AUC"

print("Holdout rows:", f"{len(test_df):,}")
print("Actual laundering cases:", f"{int(y_test.sum()):,}")
print("Fraud prevalence:", f"{y_test.mean():.4%}")
print("Date range:", times.min(), "to", times.max())

Holdout rows: 1,346,673
Actual laundering cases: 2,150
Fraud prevalence: 0.1597%
Date range: 2022-09-08 00:00:00 to 2022-09-18 16:18:00


In [13]:
scores = artifact["pipeline"].predict_proba(X_test)[:, 1]
assert len(scores) == len(y_test)
assert np.isfinite(scores).all()
assert ((scores >= 0) & (scores <= 1)).all()

flags = (scores >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, flags, labels=[0, 1]).ravel()

results = {
    "model_id": artifact["model_id"],
    "feature_version": artifact["feature_version"],
    "frozen_threshold": THRESHOLD,
    "holdout_first_timestamp": str(times.min()),
    "holdout_last_timestamp": str(times.max()),
    "transactions": int(len(y_test)),
    "actual_laundering_cases": int(y_test.sum()),
    "prevalence": float(y_test.mean()),
    "average_precision": float(average_precision_score(y_test, scores)),
    "roc_auc": float(roc_auc_score(y_test, scores)),
    "precision": float(precision_score(y_test, flags, zero_division=0)),
    "recall": float(recall_score(y_test, flags, zero_division=0)),
    "f1": float(f1_score(y_test, flags, zero_division=0)),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "alerts": int(tp + fp),
    "alert_rate": float((tp + fp) / len(y_test)),
    "false_positives_per_true_positive": float(fp / tp) if tp else None,
    "passed_80pct_recall_goal": bool(recall_score(y_test, flags) >= 0.80),
    "historical_features_manual_review": True,
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
}

print(pd.Series(results).to_string())
print("\nConfusion matrix [actual rows: legitimate, laundering]:")
print(np.array([[tn, fp], [fn, tp]]))

model_id                                                   V4-T2-B-02
feature_version                                                 V3-E1
frozen_threshold                                             0.105592
holdout_first_timestamp                           2022-09-08 00:00:00
holdout_last_timestamp                            2022-09-18 16:18:00
transactions                                                  1346673
actual_laundering_cases                                          2150
prevalence                                                   0.001597
average_precision                                            0.550584
roc_auc                                                      0.985645
precision                                                    0.016542
recall                                                        0.94186
f1                                                           0.032512
true_negatives                                                1224130
false_positives     

# Save the unmodified evaluation results

In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with RESULTS_PATH.open("x", encoding="utf-8") as f:
    json.dump(results, f, indent=2, allow_nan=False)
print("Saved final results:", RESULTS_PATH)

Saved final results: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\Results\final_test\v4_t2_b_02_final_holdout_metrics.json
